# Kernel PCA vs Standard PCA: Nonlinear Structure

**Experiment:** Generate a noisy Swiss-roll manifold and compare ordinary PCA with RBF Kernel PCA. Evaluate embeddings visually and quantitatively using reconstruction error (for standard PCA) and neighborhood preservation (trustworthiness). Kernel PCA has no simple inverse reconstruction in this setup, so comparisons focus on the embedding geometry.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import trustworthiness
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X, color = make_swiss_roll(n_samples=1500, noise=0.15, random_state=42)
X = StandardScaler().fit_transform(X)
print("Shape:", X.shape)

## 1. Visualize original data

In [ ]:
fig = plt.figure(figsize=(7,5))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(X[:,0], X[:,1], X[:,2], c=color, cmap="viridis", s=8)
ax.set_title("Swiss roll (standardized)")
plt.show()

## 2. PCA embeddings

Project into two components and inspect explained variance and inverse reconstruction error.

In [ ]:
pca = PCA(n_components=2)
Z_pca = pca.fit_transform(X)
X_rec = pca.inverse_transform(Z_pca)
pca_mse = np.mean((X-X_rec)**2)
print("PCA explained variance ratio:", pca.explained_variance_ratio_)
print("PCA reconstruction MSE:", pca_mse)

In [ ]:
plt.scatter(Z_pca[:,0], Z_pca[:,1], c=color, cmap="viridis", s=8)
plt.title("Standard PCA: 2D embedding")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.tight_layout(); plt.show()

## 3. Kernel PCA with RBF kernel

Compare a small grid of kernel bandwidth (`gamma`) values. Trustworthiness measures how well local neighborhoods are preserved in the low-dimensional embedding.

In [ ]:
gammas = [0.01, 0.05, 0.1, 0.5, 1.0]
kpca_results = {}
for gamma in gammas:
    kpca = KernelPCA(n_components=2, kernel="rbf", gamma=gamma,
                     eigen_solver="arpack", random_state=42)
    Z = kpca.fit_transform(X)
    score = trustworthiness(X, Z, n_neighbors=10)
    kpca_results[gamma] = (Z, score)
    print(f"gamma={gamma:>4}: trustworthiness={score:.4f}")

In [ ]:
fig, axes = plt.subplots(1, len(gammas), figsize=(17,4))
for ax, gamma in zip(axes, gammas):
    Z, score = kpca_results[gamma]
    ax.scatter(Z[:,0], Z[:,1], c=color, cmap="viridis", s=5)
    ax.set_title(f"RBF KPCA gamma={gamma}\nTrust={score:.3f}")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 4. Quantitative neighborhood comparison

In [ ]:
scores = {"PCA": trustworthiness(X, Z_pca, n_neighbors=10)}
scores.update({f"RBF KPCA (gamma={g})": s for g, (z,s) in kpca_results.items()})
score_table = pd.Series(scores, name="Trustworthiness (10 neighbors)").sort_values(ascending=False)
score_table

## 5. Parameter sensitivity

The RBF kernel's `gamma` controls locality: small values produce a smoother, broader similarity function, while larger values emphasize local distances. Interpret plots and scores jointly; a high trustworthiness value alone does not guarantee that all global structure is meaningful.

In [ ]:
score_table.plot(kind="bar", figsize=(10,4), title="Neighborhood preservation")
plt.ylabel("Trustworthiness")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

## Conclusion

Compare the PCA baseline with the Kernel PCA embeddings using the actual plots and trustworthiness values. Explain whether the nonlinear embedding appears to unfold the manifold more effectively, and note sensitivity to kernel bandwidth. Do not claim a universally superior method: the result depends on data geometry and parameter choice.